In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd # import panadas to read file
food_path = os.path.join(path, 'Q1_data.csv') # access to file path
data = pd.read_csv(food_path) # read the file

In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt # import to visualization
plt.figure(figsize=(10, 6))
plt.scatter(data['Distance_km'],data['Preparation_Time_min'], c=data['Delivery_Time']) # use the distance an d time as factors to spread the plots
plt.colorbar(label='dleviery time')
plt.title('Geographic Distribution of delivery time')
plt.xlabel('Distance')
plt.ylabel('Preparation_Time_min')
plt.show()

# or can just plot histogram
plt.figure(figsize=(10, 6))
plt.hist(data['Delivery_Time'], bins=60, edgecolor='black')
plt.title('Distribution of Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Count')
plt.show()


In [ ]:
# Task 1: Write your code here:
data = data.drop(columns='Order_ID', axis=1).copy()

In [ ]:
# Task 2: Write your code here:
# fill with mode for the object variables
data['Weather'] = data['Weather'].fillna(data['Weather'].mode()[0])
data['Traffic_Level'] = data['Traffic_Level'].fillna(data['Traffic_Level'].mode()[0])
data['Time_of_Day'] = data['Time_of_Day'].fillna(data['Time_of_Day'].mode()[0])
# fill with mean for the float variables
data['Courier_Experience_yrs'] = data['Courier_Experience_yrs'].fillna(data['Courier_Experience_yrs'].mean())
data['Delivery_Time'] = data['Delivery_Time'].fillna(data['Delivery_Time'].mean())


In [ ]:
# Task 3: Write your code here:
#  Do we have duplicate samples?
def check_duplicates(data):
  duplicates = data.duplicated().sum() # calculate the duplication
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    data.drop_duplicates(inplace=True) # handle the duplicate
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(data)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder [[1.0.0],[0.1.0],[0.0.1]]
from sklearn.preprocessing import LabelEncoder #import LabelEncoder as [0,1,2]
# Do we have categorical columns? # to encode
categorical_cols = data.select_dtypes(include=["object"]).columns # check for categorical types to handle it if needed

print("Categorical Columns:", list(categorical_cols)) # print the categorical columns
# we have Categorical Columns: ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
# Encode categorical columns - converts text to integers

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

data.head()

In [ ]:
# serate features and target
from sklearn.model_selection import train_test_split #for split data in % between train and test dataset
target_column = "Delivery_Time"

X = data.drop(target_column, axis=1)
y = data[target_column]


# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler as small range "best for Linear regression , logistic regression , SVM
# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # used fit to calculate the scale and transform to apply it
X_test_scaled = scaler.transform(X_test) #no fit here because it is test so we do not need a calculation

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

In [ ]:
# Task 6: Write your code here:
#it is regression ans regression check the skewed

In [ ]:
# Task 1: Write your code here:# sperate features and target
from sklearn.model_selection import KFold
target_column = "Delivery_Time"

X = data.drop(target_column, axis=1)
y = data[target_column]


# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import  mean_absolute_error #for regression metrics
import numpy as np


n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
# Storage for logistic regression results for each fold
mae_scores = []

model = RandomForestRegressor()

for train_idx, test_idx in skf.split(X_train ,X_test ):
    X_train, X_test = X_train[train_idx], X_train[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print("\nModel Evaluation Metrics ")
print(f"MAE : {np.mean(mae_scores):.2f}")

# Print the averaged score across all folds
for x in mae_scores:
  print(f"\n{x}:")
  print(f"  avg:  {np.mean(mae_scores[x]['avg']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Plot feature importance
importance = pd.DataFrame({
    'feature': X,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted dilevery time')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here: